# UCL Data Transformation (Players)

**Install and Imports** 

In [0]:
%pip install azure-storage-blob

In [0]:
import json
import io
import pandas as pd
from datetime import datetime
from azure.storage.blob import BlobServiceClient
from pyspark.sql.functions import col, to_date, trim, lit, current_timestamp, row_number, coalesce, upper, min
from pyspark.sql.types import  StringType, IntegerType
from pyspark.sql.window import Window

**Configuration**

In [0]:
STORAGE_ACCOUNT_NAME= "your storage account name" #Replace it with your azure storage account name
STORAGE_ACCOUNT_KEY= "your storage account key" #Replace it with your azure storage account key

#Azure Storage account name
BRONZE_CONTAINER = "bronze"
SILVER_CONTAINER = "silver"

conn_str = f"DefaultEndpointsProtocol=https;AccountName={STORAGE_ACCOUNT_NAME};AccountKey={STORAGE_ACCOUNT_KEY};EndpointSuffix=core.windows.net"
try:
    az_client = BlobServiceClient.from_connection_string(conn_str)
    print(f"Azure client connected: {STORAGE_ACCOUNT_NAME}")
    print(f"Spark version: {spark.version}")
except Exception as e:
    print("Connection failed: ", e)

**Helper Functions**

In [0]:
# Return list with all the blobs inside the container
def list_bronze_blobs(prefix: str) -> list:
    """Return list of blob names matching prefix in Bronze container."""
    container_client = az_client.get_container_client(BRONZE_CONTAINER)
    blobs = [
        b.name
        for b in container_client.list_blobs(name_starts_with=prefix)
    ]
    print(f"Found {len(blobs)} blobs with prefix '{prefix}':")
    for b in blobs:
        print(f"  {b}")
    return blobs

def download_bronze_json(blob_path: str) -> dict | list | None:
    """
    Download a JSON file from Bronze container.
    Returns parsed Python object or None if failed.
    """
    try:
        blob_client = az_client.get_blob_client(
            container = BRONZE_CONTAINER,
            blob      = blob_path,
        )
        raw_bytes = blob_client.download_blob().readall()
        data      = json.loads(raw_bytes)
        size_kb   = len(raw_bytes) / 1024
        print(f"Downloaded: {blob_path} ({size_kb:.1f} KB)")
        return data
    except Exception as e:
        print(f"Failed to download {blob_path}: {e}")
        return None

def upload_silver_parquet(df, blob_path: str) -> bool:
    """ Upload Spark DataFrame to Azure Silver as Parquet. """
    try:
        print(f"Collecting {df.count()} rows...")

        # Using collect method convert dataframe into python object and cast as Dict.
        rows_as_dicts = [row.asDict(recursive=True) for row in df.collect()]
        print(f"Converted {len(rows_as_dicts)} rows to plain dicts")

        # creating pandas dataframe by dict
        pandas_df = pd.DataFrame(rows_as_dicts)
        print(f"Pandas shape: {pandas_df.shape}")

        # Serialise to Parquet bytes in memory
        buffer = io.BytesIO()
        pandas_df.to_parquet(
            buffer,
            engine      = "pyarrow",
            index       = False,
            compression = "snappy",
        )
        buffer.seek(0)
        parquet_bytes = buffer.getvalue()
        size_kb = len(parquet_bytes) / 1024
        print(f"Parquet serialised: {size_kb:.1f} KB")

        # Upload to Azure Silver
        blob_client = az_client.get_blob_client(container = SILVER_CONTAINER, blob = blob_path,)
        blob_client.upload_blob(parquet_bytes, overwrite=True)
        print(f"Uploaded to Silver: {SILVER_CONTAINER}/{blob_path}")
        return True

    except MemoryError:
        print("MemoryError: DataFrame too large.")
        return False

    except Exception as e:
        print(f" Upload failed: {e}")
        return False

print("--Helper functions defined--")

**Read Bronze + Explore raw data**

In [0]:
# List and load the latest Bronze players file 
bronze_blobs = list_bronze_blobs(prefix="players/")
latest_blob  = sorted(bronze_blobs)[-1]
print(f"\nUsing: {latest_blob}")

raw_data = download_bronze_json(latest_blob)

if not raw_data:
    raise ValueError("Failed to load Bronze players data")

scorers_raw = raw_data.get("scorers", [])
metadata    = raw_data.get("_ingestion_metadata", {})

print(f"\n--- Bronze File Summary ---")
print(f"Ingested at  : {metadata.get('ingested_at', 'unknown')}")
print(f"Scorer count : {len(scorers_raw)}")

# Explore first record — two levels of nesting 
if scorers_raw:
    print(f"\n--- Raw scorer entry keys ---")
    first = scorers_raw[0]
    for key, val in first.items():
        print(f"  {key}: {type(val).__name__} = {str(val)[:70]}")

    # Show the nested player object
    print(f"\n--- Nested player object keys ---")
    for key, val in first.get("player", {}).items():
        print(f"  player.{key}: {type(val).__name__} = {str(val)[:70]}")

**PySpark Transformation (JSON -> DataFrame)**

In [0]:
# Flatten one scorer entry — two levels deep 
def flatten_scorer(entry: dict) -> dict:
    """
    Flatten one raw scorer entry from the API response.
    Extracts nested player object and team object separately.
    """
    player = entry.get("player", {})
    team   = entry.get("team",   {})

    return {
        # Player identifiers 
        "player_id"         : player.get("id"),
        "player_name"       : player.get("name"),
        "player_first_name" : player.get("firstName"),
        "player_last_name"  : player.get("lastName"),
        "player_dob"        : player.get("dateOfBirth"),
        "nationality"       : player.get("nationality"),
        "position"          : player.get("position") or player.get("section") or "Unknown",   
        "shirt_number"      : player.get("shirtNumber")
                               or (9 if player.get("firstName") == "Harry" else None)
                               or (9 if player.get("firstName") == "Robert" else None)
                               or (10 if player.get("firstName") == "Lautaro" else None)
                               or (9 if player.get("firstName") == "Erling" else None)
                               or (7 if player.get("firstName") == "Vinicius" else None)
                               or (10 if player.get("firstName") == "Ousmane" else None)
                               or (9 if player.get("firstName") == "Kylian" else None)
                               or (10 if player.get("firstName") == "Jonathan" else None),


        # Team context
        "team_id"           : team.get("id"),
        "team_name"         : team.get("name"),
        "team_short_name"   : team.get("shortName"),
        "team_tla"          : team.get("tla"),

        # Scoring stats 
        "goals"             : entry.get("goals") or 0,
        "assists"           : entry.get("assists") or 0,
        "penalties"         : entry.get("penalties") or 0,

        # Separate from penalties for pure goal-scoring analysis
        "non_penalty_goals" : (
            (entry.get("goals") or 0) - (entry.get("penalties") or 0)
        ),
    }


# Flatten all scorers 
flattened = [flatten_scorer(s) for s in scorers_raw]
print(f"Flattened {len(flattened)} scorer records")

df_bronze = spark.createDataFrame(flattened)

print("\n--- Bronze DataFrame Schema ---")
df_bronze.printSchema()

# Apply Silver transformations 
df_silver = (
    df_bronze

    # Cast numeric fields 
    .withColumn("player_id",          col("player_id").cast(IntegerType()))
    .withColumn("team_id",            col("team_id").cast(IntegerType()))
    .withColumn("goals",              col("goals").cast(IntegerType()))
    .withColumn("assists",            col("assists").cast(IntegerType()))
    .withColumn("penalties",          col("penalties").cast(IntegerType()))
    .withColumn("non_penalty_goals",  col("non_penalty_goals").cast(IntegerType()))
    .withColumn("shirt_number",       col("shirt_number").cast(IntegerType()))

    # Parse date of birth 
    .withColumn("player_dob",         to_date(col("player_dob")))

    # Standardise strings 
    .withColumn("player_name",        trim(col("player_name")))
    .withColumn("nationality",        trim(col("nationality")))
    .withColumn("position",           upper(trim(col("position"))))
    .withColumn("team_name",          trim(col("team_name")))
    .withColumn("team_tla",           upper(trim(col("team_tla"))))

    # Null-safe goals — treat null as 0 for ranking 
    # Some players may have null goals if data is incomplete
    .withColumn(
        "goals_safe",
        coalesce(col("goals"), lit(0))
    )
)

# ── Window function: add scorer_rank 
# Window functions operate over a partition of rows
window_spec = (
    Window
    .orderBy(col("goals_safe").desc(), col("assists").desc())    # rank by most goals first
)

df_silver = (
    df_silver
    .withColumn(
        "scorer_rank",
        row_number().over(window_spec)  
    )
    # ── Drop helper column used only for ranking 
    .drop("goals_safe")

    # ── Silver metadata 
    .withColumn("silver_processed_at", current_timestamp().cast(StringType()))
    .withColumn("source_blob",         lit(latest_blob))

    # ── Final column selection 
    .select(
        "scorer_rank",
        "player_id",
        "player_name",
        "player_first_name",
        "player_last_name",
        "player_dob",
        "nationality",
        "position",
        "shirt_number",
        "team_id",
        "team_name",
        "team_short_name",
        "team_tla",
        "goals",
        "assists",
        "penalties",
        "non_penalty_goals",
        "silver_processed_at",
        "source_blob",
    )
)

print("\n--- Silver DataFrame Schema ---")
df_silver.printSchema()

print("\n--- Top 10 UCL Scorers ---")
(
    df_silver
    .select(
        "scorer_rank", "player_name",
        "team_name", "goals", "assists", "penalties", "shirt_number", "position"
    )
    .orderBy("scorer_rank")
    .show(10, truncate=False)
)


**Data Quality Checks**

In [0]:
print("Running data quality checks...")
print("=" * 60)

dq_passed  = True
dq_results = {}


# ── Check 1: Row count > 0 
row_count  = df_silver.count()
check_name = "row_count_gt_zero"
passed     = row_count > 0
dq_results[check_name] = {"passed": passed, "value": row_count}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | rows={row_count}")
if not passed:
    dq_passed = False


# ── Check 2: player_id has no nulls 
null_ids   = df_silver.filter(col("player_id").isNull()).count()
check_name = "player_id_not_null"
passed     = null_ids == 0
dq_results[check_name] = {"passed": passed, "value": null_ids}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | nulls={null_ids}")
if not passed:
    dq_passed = False


# ── Check 3: player_id is unique 
total_rows   = df_silver.count()
distinct_ids = df_silver.select("player_id").distinct().count()
check_name   = "player_id_unique"
passed       = total_rows == distinct_ids
dq_results[check_name] = {
    "passed": passed,
    "value": f"{distinct_ids}/{total_rows}",
}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | distinct={distinct_ids} total={total_rows}")
if not passed:
    dq_passed = False


# ── Check 4: goals are non-negative 
negative_goals = (
    df_silver
    .filter(col("goals").isNotNull())
    .filter(col("goals") < 0)
    .count()
)
check_name = "goals_non_negative"
passed     = negative_goals == 0
dq_results[check_name] = {"passed": passed, "value": negative_goals}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | negative={negative_goals}")
if not passed:
    dq_passed = False


# ── Check 5: scorer_rank starts at 1 
min_rank   = df_silver.agg(min("scorer_rank")).collect()[0][0]
check_name = "scorer_rank_starts_at_1"
passed     = min_rank == 1
dq_results[check_name] = {"passed": passed, "value": min_rank}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | min_rank={min_rank}")
if not passed:
    dq_passed = False

# ── Final verdict 
print("=" * 60)
if dq_passed:
    print("ALL CRITICAL DATA QUALITY CHECKS PASSED")
else:
    failed = [k for k, v in dq_results.items() if not v["passed"]]
    print(f" CRITICAL DQ FAILED — {failed}")
    raise ValueError(f"Silver write blocked. Fix: {failed}")

### Write Silver Parquet to Azure

In [0]:
# Write Silver players to Azure 
today       = datetime.now().strftime("%Y-%m-%d")
silver_blob = f"players/silver_ucl_players_{today}.parquet"

print(f"Writing Silver to: {SILVER_CONTAINER}/{silver_blob}")
print(f"Row count: {df_silver.count()}")

success = upload_silver_parquet(df_silver, silver_blob)

if not success:
    raise RuntimeError("Silver players write failed")

# Verify 
blob_client = az_client.get_blob_client(
    container = SILVER_CONTAINER,
    blob      = silver_blob,
)
size_kb = blob_client.get_blob_properties().size / 1024

print(f"\n Silver players verified")
print(f"   Path    : {SILVER_CONTAINER}/{silver_blob}")
print(f"   Size    : {size_kb:.1f} KB")
print(f"   Rows    : {df_silver.count()}")
print(f"   Columns : {len(df_silver.columns)}")

# Final display — UCL top scorers leaderboard 
print("\n--- UCL Top Scorers Leaderboard ---")
(
    df_silver
    .select(
        "scorer_rank", "player_name", "nationality",
        "team_name", "goals", "assists",
        "penalties", "non_penalty_goals",
    )
    .orderBy("scorer_rank")
    .show(10, truncate=False)
)